In [1]:
# %%
"""
=========================================================================================
PROJECT: LANDSLIDE FORECASTING - CNN-LSTM HYBRID (Balanced)
=========================================================================================
[ARCHITECTURE]
1. CNN Layer: สกัดฟีเจอร์เด่น (Feature Extraction) ลด Noise และจับ Pattern การกระชาก
2. LSTM Layer: วิเคราะห์ลำดับเวลา (Temporal Dependency) จากฟีเจอร์ที่ CNN ส่งมา

[DATA STRATEGY]
- Multi-Horizon Forecast (0m - 24h)
- Advanced Balancing (Up/Down Sampling)
- Safety Class Weights
"""

import os
import glob
import time
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, Bidirectional, Conv1D, MaxPooling1D, Flatten, TimeDistributed
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# 1. CONFIGURATION
# ====================================================================
class Config:
    DATA_DIR = "./data/"
    MODEL_DIR = "./model/cnn_lstm_balanced/"
    SCALER_PATH = "./model/cnn_lstm_scaler.save"
    
    # Horizons
    HORIZONS = [0, 30, 60, 180, 360, 720, 1440]
    
    # Settings
    SEQUENCE_LENGTH = 60  # 60 นาที
    TEST_KEYWORDS = ['label_for_dev109_test_prepared']
    
    RAW_COLS = ['rain', 'soil', 'temp', 'humi', 'geo']
    LABEL_COL = 'label'
    LABEL_MAP = {'normal': 0, 'warning': 1, 'critical': 2}
    
    # --- BALANCING ---
    TARGET_SAMPLES = 5000 # ปรับให้ทุกคลาสมีเท่านี้
    
    # Training
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 1e-3
    
    # Weights (ยังใส่ไว้เป็น Safety Net แม้จะ Balance แล้ว)
    CLASS_WEIGHTS = {0: 1.0, 1: 3.0, 2: 5.0}

cfg = Config()
if not os.path.exists(cfg.MODEL_DIR): os.makedirs(cfg.MODEL_DIR)
np.random.seed(42); tf.random.set_seed(42); random.seed(42)

def log(msg): print(f"[INFO] {time.strftime('%H:%M:%S')} - {msg}")
print("✅ CNN-LSTM Config Loaded.")

# ====================================================================
# 2. LOAD & FEATURE ENGINEERING
# ====================================================================
def load_data():
    log("Scanning Data...")
    files = glob.glob(os.path.join(cfg.DATA_DIR, "*.csv"))
    if not files: files = glob.glob(os.path.join(cfg.DATA_DIR, "**/*.csv"), recursive=True)
    
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            df.columns = [str(c).lower().strip().replace('.1', '') for c in df.columns]
            rename_map = {'temperature':'temp', 'hum':'humi', 'humidity':'humi', 'devid':'devID', 'time':'timestamp', 'date':'timestamp'}
            new_cols = {c: rename_map[c] for c in df.columns if c in rename_map}
            if new_cols: df = df.rename(columns=new_cols)
            
            if 'devID' in df.columns: df['devID'] = df['devID'].astype(str).str.extract('(\d+)')[0].astype(float).fillna(0).astype(int)
            if 'timestamp' in df.columns: df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            for c in cfg.RAW_COLS: 
                if c not in df.columns: df[c] = 0.0
            
            filename = os.path.basename(f)
            df['is_test'] = any(k in filename for k in cfg.TEST_KEYWORDS)
            dfs.append(df)
        except: pass
    return pd.concat(dfs, ignore_index=True) if dfs else None

def generate_features(df):
    log("Generating Features...")
    df = df.sort_values(['devID', 'timestamp']).reset_index(drop=True)
    df_list = []
    
    for dev, g in df.groupby('devID'):
        g = g.set_index('timestamp')
        g = g[~g.index.duplicated(keep='first')]
        if len(g) > 0: g = g.resample('1T').asfreq()
        g[cfg.RAW_COLS] = g[cfg.RAW_COLS].interpolate(limit_direction='both').fillna(0)
        
        # Physics Features
        g['rain_1h'] = g['rain'].rolling(60, min_periods=1).sum()
        g['soil_rate'] = g['soil'].diff().fillna(0)
        g['soil_acc']  = g['soil'].diff().diff().fillna(0)
        g['rain_6h'] = g['rain'].rolling(360, min_periods=1).sum()
        g['soil_trend_6h'] = g['soil'] - g['soil'].shift(360).fillna(method='bfill')
        g['rain_24h'] = g['rain'].rolling(1440, min_periods=1).sum()
        g['soil_max_24h'] = g['soil'].rolling(1440, min_periods=1).max()
        g['rain_x_soil'] = g['rain'] * g['soil']
        g['rain_x_geo']  = g['rain'] * g['geo'].abs()
        
        if cfg.LABEL_COL in g.columns:
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].fillna('normal').astype(str).str.lower().str.strip()
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].map(cfg.LABEL_MAP).fillna(0).astype(int)
        else: g[cfg.LABEL_COL] = 0
        
        g['is_test'] = g['is_test'].fillna(method='ffill').fillna(method='bfill')
        g['devID'] = dev
        df_list.append(g.reset_index())
        
    return pd.concat(df_list, ignore_index=True).fillna(0)

df_train_raw = load_data()
df_proc = generate_features(df_train_raw)

# ====================================================================
# 3. SEQUENCE & BALANCING
# ====================================================================
def create_sequences(df, feature_cols, horizon):
    Xs, ys, is_tests = [], [], []
    for dev, g in df.groupby('devID'):
        data = g[feature_cols].values
        labels = g[cfg.LABEL_COL].shift(-horizon).values
        test_flag = g['is_test'].values
        
        valid_len = len(g) - horizon
        if valid_len < cfg.SEQUENCE_LENGTH: continue
        
        for i in range(valid_len - cfg.SEQUENCE_LENGTH):
            Xs.append(data[i : i+cfg.SEQUENCE_LENGTH])
            ys.append(labels[i+cfg.SEQUENCE_LENGTH])
            is_tests.append(test_flag[i+cfg.SEQUENCE_LENGTH])
            
    return np.array(Xs), np.array(ys), np.array(is_tests)

def balance_data(X, y):
    """
    Smart Balancing: Oversample Minority + Downsample Majority
    """
    print(f"   Original counts: {Counter(y)}")
    X_bal, y_bal = [], []
    
    for cls in np.unique(y):
        if np.isnan(cls): continue
        indices = np.where(y == cls)[0]
        count = len(indices)
        
        # ถ้าข้อมูลน้อยกว่าเป้า -> สุ่มซ้ำ (Replace=True)
        # ถ้าข้อมูลเยอะกว่าเป้า -> สุ่มออก (Replace=False)
        replace = count < cfg.TARGET_SAMPLES
        
        chosen_indices = np.random.choice(
            indices, 
            cfg.TARGET_SAMPLES, 
            replace=replace
        )
        
        X_bal.append(X[chosen_indices])
        y_bal.append(y[chosen_indices])
        
    X_bal = np.concatenate(X_bal)
    y_bal = np.concatenate(y_bal)
    
    # Shuffle
    perm = np.random.permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]

# ====================================================================
# 4. CNN-LSTM ARCHITECTURE
# ====================================================================
def build_cnn_lstm(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    
    # --- Part 1: CNN (Feature Extractor) ---
    # สแกนหา Pattern ช่วงสั้นๆ (เช่น 3 นาที)
    x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # ย่อข้อมูลลงครึ่งนึง (60 -> 30 timesteps)
    x = Dropout(0.3)(x)
    
    x = Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # ย่อข้อมูลลงอีก (30 -> 15 timesteps)
    x = Dropout(0.3)(x)
    
    # --- Part 2: LSTM (Sequence Modeling) ---
    # รับข้อมูลที่ถูกย่อและสกัดเนื้อๆ มาแล้วจาก CNN
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # --- Part 3: Output ---
    x = Dense(64, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name="CNN_LSTM_Hybrid")
    model.compile(optimizer=Adam(learning_rate=cfg.LEARNING_RATE), 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

# ====================================================================
# 5. TRAINING LOOP
# ====================================================================
def train_system():
    FINAL_FEATURES = cfg.RAW_COLS + ['rain_1h', 'soil_rate', 'soil_acc', 'rain_6h', 'soil_trend_6h', 'rain_24h', 'soil_max_24h', 'rain_x_soil', 'rain_x_geo']
    
    log("Scaling Data...")
    scaler = RobustScaler()
    train_mask = ~df_proc['is_test']
    scaler.fit(df_proc.loc[train_mask, FINAL_FEATURES])
    df_proc[FINAL_FEATURES] = scaler.transform(df_proc[FINAL_FEATURES])
    joblib.dump(scaler, cfg.SCALER_PATH)
    
    results_log = []

    for h in cfg.HORIZONS:
        print(f"\n{'='*60}")
        print(f"🚀 CNN-LSTM HORIZON: +{h/60:.1f} Hours ({h} mins)")
        print(f"{'='*60}")
        
        # 1. Create Sequences
        X_all, y_all, test_mask = create_sequences(df_proc, FINAL_FEATURES, h)
        
        X_train = X_all[~test_mask]; y_train = y_all[~test_mask]
        X_test  = X_all[test_mask];  y_test  = y_all[test_mask]
        
        valid_tr = ~np.isnan(y_train); X_train = X_train[valid_tr]; y_train = y_train[valid_tr]
        valid_ts = ~np.isnan(y_test);  X_test = X_test[valid_ts];   y_test = y_test[valid_ts]
        
        # 2. Split Val
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
        
        # 3. BALANCE TRAIN (Oversample/Downsample)
        print("⚖️ Balancing Train Data...")
        X_train_bal, y_train_bal = balance_data(X_train, y_train)
        print(f"   Balanced Counts: {Counter(y_train_bal)}")
        
        # Encode
        y_train_cat = to_categorical(y_train_bal, 3)
        y_val_cat = to_categorical(y_val, 3)
        
        # Weights
        safety_factor = 1.0 + (h/1440.0)
        cw = {k: v * safety_factor for k, v in cfg.CLASS_WEIGHTS.items()}
        
        # 4. Train
        model = build_cnn_lstm((cfg.SEQUENCE_LENGTH, len(FINAL_FEATURES)), 3)
        save_path = os.path.join(cfg.MODEL_DIR, f"cnnlstm_h{h}.h5")
        
        cbs = [
            EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
            ReduceLROnPlateau(patience=5, factor=0.5, monitor='val_loss'),
            ModelCheckpoint(save_path, save_best_only=True, monitor='val_accuracy', verbose=0)
        ]
        
        model.fit(X_train_bal, y_train_cat, validation_data=(X_val, y_val_cat),
                  epochs=cfg.EPOCHS, batch_size=cfg.BATCH_SIZE, 
                  class_weight=cw, callbacks=cbs, verbose=1)
        
        # 5. Evaluate
        probs = model.predict(X_test, verbose=0)
        y_pred = np.argmax(probs, axis=1)
        
        report = classification_report(y_test, y_pred, target_names=['Normal', 'Warning', 'Critical'], output_dict=True, zero_division=0)
        f1_norm = report['Normal']['f1-score']
        f1_warn = report['Warning']['f1-score']
        f1_crit = report['Critical']['f1-score']
        acc = accuracy_score(y_test, y_pred)
        
        print(f"   📊 Accuracy : {acc:.4f}")
        print(f"   🟢 F1-Norm  : {f1_norm:.4f}")
        print(f"   🟡 F1-Warn  : {f1_warn:.4f}")
        print(f"   🔴 F1-Crit  : {f1_crit:.4f}")
        
        results_log.append({'Horizon': h, 'Acc': acc, 'F1_Norm': f1_norm, 'F1_Warn': f1_warn, 'F1_Crit': f1_crit})

    summary = pd.DataFrame(results_log)
    summary.to_csv("./model/cnnlstm_horizon_performance.csv", index=False)
    print("\n=== FINAL SUMMARY ===")
    print(summary)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(summary['Horizon'], summary['F1_Crit'], marker='o', color='red', label='Critical F1')
    plt.plot(summary['Horizon'], summary['F1_Warn'], marker='s', color='orange', label='Warning F1')
    plt.title("CNN-LSTM Forecast Performance")
    plt.xlabel("Minutes Ahead")
    plt.ylabel("F1 Score")
    plt.legend(); plt.grid(True)
    plt.show()

if __name__ == "__main__":
    train_system()

✅ CNN-LSTM Config Loaded.
[INFO] 01:01:03 - Scanning Data...
[INFO] 01:01:04 - Generating Features...
[INFO] 01:01:04 - Scaling Data...

🚀 CNN-LSTM HORIZON: +0.0 Hours (0 mins)
⚖️ Balancing Train Data...
   Original counts: Counter({np.int64(0): 24393, np.int64(1): 423, np.int64(2): 252})
   Balanced Counts: Counter({np.int64(2): 5000, np.int64(0): 5000, np.int64(1): 5000})
Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.6141 - loss: 2.2507

235/235 ━━━━━━━━━━━━━━━━━━━━ 76s 148ms/step - accuracy: 0.6144 - loss: 2.2492 - val_accuracy: 0.0190 - val_loss: 2.2908 - learning_rate: 0.0010
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.7405 - loss: 1.4857

235/235 ━━━━━━━━━━━━━━━━━━━━ 27s 112ms/step - accuracy: 0.7405 - loss: 1.4853 - val_accuracy: 0.2923 - val_loss: 1.1128 - learning_rate: 0.0010
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 43s 117ms/step - accuracy: 0.7875 - loss: 1.1828 - val_accuracy: 0.0688 - val_loss: 1.2547 - learning_rate: 0.0010
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.8142 - loss: 0.9981

235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 128ms/step - accuracy: 0.8142 - loss: 0.9979 - val_accuracy: 0.7115 - val_loss: 0.7866 - learning_rate: 0.0010
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.8452 - loss: 0.8726

235/235 ━━━━━━━━━━━━━━━━━━━━ 41s 126ms/step - accuracy: 0.8452 - loss: 0.8726 - val_accuracy: 0.7330 - val_loss: 0.7347 - learning_rate: 0.0010
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 36s 103ms/step - accuracy: 0.8561 - loss: 0.7974 - val_accuracy: 0.7101 - val_loss: 0.8506 - learning_rate: 0.0010
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.8630 - loss: 0.7631

...

235/235 ━━━━━━━━━━━━━━━━━━━━ 22s 93ms/step - accuracy: 0.9670 - loss: 0.1696 - val_accuracy: 0.9244 - val_loss: 0.3101 - learning_rate: 5.0000e-04
Epoch 34/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 41s 90ms/step - accuracy: 0.9693 - loss: 0.1590 - val_accuracy: 0.9157 - val_loss: 0.3561 - learning_rate: 5.0000e-04
   📊 Accuracy : 0.7775
   🟢 F1-Norm  : 0.8805
   🟡 F1-Warn  : 0.4070
   🔴 F1-Crit  : 0.5658

🚀 CNN-LSTM HORIZON: +0.5 Hours (30 mins)
⚖️ Balancing Train Data...
   Original counts: Counter({np.float64(0.0): 24246, np.float64(1.0): 436, np.float64(2.0): 266})
   Balanced Counts: Counter({np.float64(1.0): 5000, np.float64(0.0): 5000, np.float64(2.0): 5000})
Epoch 1/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.5268 - loss: 2.6580

235/235 ━━━━━━━━━━━━━━━━━━━━ 30s 80ms/step - accuracy: 0.5272 - loss: 2.6554 - val_accuracy: 0.0297 - val_loss: 2.3106 - learning_rate: 0.0010
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 23s 96ms/step - accuracy: 0.6426 - loss: 2.0331 - val_accuracy: 0.0200 - val_loss: 2.1339 - learning_rate: 0.0010
Epoch 3/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.6876 - loss: 1.7721

235/235 ━━━━━━━━━━━━━━━━━━━━ 25s 107ms/step - accuracy: 0.6876 - loss: 1.7718 - val_accuracy: 0.0369 - val_loss: 1.7404 - learning_rate: 0.0010
Epoch 4/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.7260 - loss: 1.5642

235/235 ━━━━━━━━━━━━━━━━━━━━ 39s 96ms/step - accuracy: 0.7261 - loss: 1.5639 - val_accuracy: 0.6288 - val_loss: 1.0715 - learning_rate: 0.0010
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 49s 126ms/step - accuracy: 0.7469 - loss: 1.4032 - val_accuracy: 0.1004 - val_loss: 1.3440 - learning_rate: 0.0010
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 29s 123ms/step - accuracy: 0.7747 - loss: 1.2405 - val_accuracy: 0.2509 - val_loss: 1.4146 - learning_rate: 0.0010
Epoch 7/100
221/235 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - accuracy: 0.7881 - loss: 1.1661

KeyboardInterrupt: 